In [ ]:
from transformers import PatchTSTConfig, PatchTSTForPretraining
import torch
import numpy as np
import time

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model_name = 'PE-TSFM-25M16P'

dim = 512
layers = 8

config = PatchTSTConfig(
            num_input_channels=8,
            context_length=512,
            patch_length=16,
            patch_stride=16,
            num_targets=4,
            num_hidden_layers=layers,
            d_model=dim,
            num_attention_heads=dim//64,
            ffn_dim=dim*4,
            head_dropout=0.2,
            ff_dropout=0.2,
            attention_dropout=0.2,
            use_cls_token=False,
            share_embedding=False,
            pre_norm=False,
            channel_attention=True,
            pooling_type='mean',
            mask_type='random',
            random_mask_ratio=0.3,
        )

model = PatchTSTForPretraining(config).to('cuda')
print(model)
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {num_params/1e6:.1f} M")

In [ ]:
from load import *
from torch.utils.data import DataLoader, TensorDataset
import torch

x_train, _ = load_data(folder_path='../PCT/train', seq_length=512, stride=512)

train_dataset = TensorDataset(torch.tensor(x_train, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

print('The number of training samples: ', x_train.shape[0])
print('The shape of each sample: ', x_train.shape[1:])

In [ ]:
from trainer import Trainer
from WarmupCosineDecay import WarmupCosineDecay
import torch.optim as optim

epochs = 10
batchs = len(train_loader)
total_steps = epochs * batchs
warmup_steps = int(0.1 * total_steps)
decay_steps = total_steps - warmup_steps

optim = optim.AdamW(model.parameters())

print(f"Total pretrain steps: {total_steps}, Warmup steps: {warmup_steps}, Decay steps: {decay_steps}")

scheduler = WarmupCosineDecay(
    optimizer=optim,
    init_value=1e-6,
    peak_value=2e-4,
    warmup_steps=warmup_steps,
    decay_steps=decay_steps
)

trainer = Trainer(model,
                  lr_scheduler=scheduler,
                  optimizer=optim,
                  max_epochs=epochs,
                  use_early_stopping=False, 
                  use_amp=True,
                  device='cuda' if torch.cuda.is_available() else 'cpu',
                  model_name=model_name + '-' + time.strftime("%Y%m%d-%H%M%S")
                  )

In [ ]:
trainer.pretrain(train_loader)
model.save_pretrained("./models/" + model_name)
print(f"Model saved to ./models/{model_name}")